# 06 — Trade Lifecycle and Exit Taxonomy

**Strategy reference:** §12 (Lifecycle), §13 (Exits).

Every trade walks the state machine:

```
SETUP → ENTRY → CONFIRMATION → EXPANSION → MATURATION → EXIT → COOLDOWN
                                ↑              ↓
                              EXIT (any)    EXIT (any)
```

Risk-budget and permission-revocation exits override everything
from any active state.

In [ ]:
# ── Data-source configuration ─────────────────────────────────────────
# OHLCV (Parquet) — S3 or local, controlled by DATA_STORE env var:
#   Local (default):  reads <project_root>/data/ohlcv/...
#   S3:               uncomment the two lines below
# import os
# os.environ["DATA_STORE"] = "s3"
# os.environ["S3_BUCKET"]  = "trading-data-centheos"
#
# Tick data (HDF5) — always stored locally; pull from S3 on demand:
#   load_ticks(...)              → use local cache (fast, no network)
#   load_ticks(..., refresh=True) → sync from S3 then read (ETag-gated)
#   Requires: AWS_PROFILE=trading (or AWS_ACCESS_KEY_ID / AWS_SECRET_ACCESS_KEY)
import os
os.environ["AWS_PROFILE"] = "trading"
os.environ["S3_BUCKET"]   = "trading-data-centheos"
# ─────────────────────────────────────────────────────────────────────

import sys, importlib
from pathlib import Path

_here = Path.cwd().resolve()
for _cand in [_here, *_here.parents]:
    if (_cand / "schemas.py").exists():
        _root = _cand; break
else:
    raise RuntimeError("Could not locate project root (no schemas.py found)")
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

import notebooks.utils as _utils_mod
importlib.reload(_utils_mod)   # always pick up on-disk changes without restarting the kernel

from notebooks.utils import (
    load_ohlcv, list_ohlcv, load_ticks, latest_book,
    plot_ohlcv, plot_equity_curve, configure_pandas, env_summary,
)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

configure_pandas()
%matplotlib inline

In [ ]:
from schemas import LifecycleState, ExitType, TradeArchetype, TradeSide, RippleConfig
cfg = RippleConfig()
[s.value for s in LifecycleState]

## 1. Exit taxonomy & priority (§13)
Evaluation order — first match fires:

1. **Risk-budget**   — `consumed_es >= 0.9 × budget` OR `risk_multiplier == 0`
2. **Invalidation**  — stop hit (bounce: wall breaks; breakout: wall re-forms)
3. **Time**          — `hold_time > max_hold_time_ms`
4. **Target**        — price ≥ `current_target_price` (favourable)
5. **Exhaustion**    — flow dying + rate down + impact rising


In [ ]:
priority_table = pd.DataFrame([
    {'priority': 1, 'exit': 'RISK_BUDGET',  'order_type': 'MARKET', 'qty': '100%'},
    {'priority': 2, 'exit': 'INVALIDATION', 'order_type': 'MARKET', 'qty': '100%'},
    {'priority': 3, 'exit': 'TIME',         'order_type': 'MARKET', 'qty': '100%'},
    {'priority': 4, 'exit': 'TARGET',       'order_type': 'LIMIT',  'qty': 'next scale-out fraction'},
    {'priority': 5, 'exit': 'EXHAUSTION',   'order_type': 'LIMIT',  'qty': '100% of remainder'},
])
priority_table

## 2. Implement each exit checker
These mirror §13.3 line-by-line.

In [ ]:
from dataclasses import dataclass, field

@dataclass
class TradeCtx:
    archetype: str            # 'BOUNCE' | 'BREAKOUT'
    side: str                 # 'LONG' | 'SHORT'
    side_sign: int
    entry_price: float
    entry_ts: int
    wall_price: float
    current_target_price: float
    entry_trade_rate: float
    entry_impact: float

@dataclass
class RippleCtx:
    timestamp: int
    last_trade_price: float
    microprice: float
    cvd_slope: float
    trade_rate: float
    impact: float
    nearest_wall_depth: float
    nearest_wall_quality: float

def check_invalidation(trade, ripple, cfg):
    if trade.archetype == 'BOUNCE':
        price_through = (ripple.last_trade_price - trade.wall_price) * (-trade.side_sign) > 0
        wall_gone = ripple.nearest_wall_depth < cfg.breakout_depth_fail_abs
        return price_through and wall_gone
    if trade.archetype == 'BREAKOUT':
        price_reversed = (trade.wall_price - ripple.last_trade_price) * trade.side_sign > 0
        wall_reformed = ripple.nearest_wall_quality > cfg.wall_min_quality
        return price_reversed and wall_reformed
    return False

def check_target(trade, ripple):
    return (ripple.microprice - trade.current_target_price) * trade.side_sign >= 0

def check_exhaustion(trade, ripple, cfg):
    cvd_fading   = ripple.cvd_slope * trade.side_sign < cfg.exhaustion_cvd_slope_thresh
    rate_falling = ripple.trade_rate < trade.entry_trade_rate * 0.5
    impact_up    = ripple.impact > trade.entry_impact * cfg.exhaustion_impact_ratio
    return cvd_fading and rate_falling and impact_up

def check_time(trade, ripple, cfg):
    return ripple.timestamp - trade.entry_ts > cfg.max_hold_time_ms

def check_risk_budget(consumed_es, es_budget, risk_multiplier, threshold=0.90):
    return consumed_es / max(es_budget, 1e-9) >= threshold or risk_multiplier == 0.0

## 3. Walk a synthetic trade through the lifecycle
Generate a price path, advance the lifecycle, capture state
transitions and which exit ultimately fires.

In [ ]:
rng = np.random.default_rng(42)
ticks_n = 600
drift   = 0.04
vol     = 0.6
prices = 100 + np.cumsum(rng.normal(drift, vol, ticks_n))
ts     = np.arange(ticks_n) * 1_000  # 1 sec per tick

trade = TradeCtx(
    archetype='BOUNCE', side='LONG', side_sign=+1,
    entry_price=prices[0], entry_ts=int(ts[0]),
    wall_price=prices[0] - 1.5, current_target_price=prices[0] + 3.0,
    entry_trade_rate=10.0, entry_impact=0.01,
)

state = 'CONFIRMATION'
history = []
exit_type = None
expansion_threshold_units = cfg.expand_threshold_sigma * 1.0  # σ=1 for synthetic

for t, p in zip(ts, prices):
    ctx = RippleCtx(
        timestamp=int(t), last_trade_price=float(p), microprice=float(p),
        cvd_slope=0.03 - 0.0002 * (t / 1_000),
        trade_rate=max(2.0, 10.0 - 0.02 * (t / 1_000)),
        impact=0.01 + 0.00002 * (t / 1_000),
        nearest_wall_depth=5.0 if state != 'EXIT' else 0.0,
        nearest_wall_quality=0.6,
    )
    if check_risk_budget(consumed_es=0.0, es_budget=1.0, risk_multiplier=1.0):
        exit_type = 'RISK_BUDGET'; state = 'EXIT'
    elif check_invalidation(trade, ctx, cfg):
        exit_type = 'INVALIDATION'; state = 'EXIT'
    elif check_time(trade, ctx, cfg):
        exit_type = 'TIME'; state = 'EXIT'
    elif check_target(trade, ctx):
        exit_type = 'TARGET'; state = 'EXIT'
    elif check_exhaustion(trade, ctx, cfg):
        exit_type = 'EXHAUSTION'; state = 'EXIT'
    else:
        favourable = (p - trade.entry_price) * trade.side_sign
        if state == 'CONFIRMATION' and favourable > expansion_threshold_units:
            state = 'EXPANSION'
        elif state == 'EXPANSION' and favourable > expansion_threshold_units * 2:
            state = 'MATURATION'
    history.append({'ts': t, 'price': p, 'state': state, 'exit': exit_type})
    if state == 'EXIT':
        break

trace = pd.DataFrame(history)
print(f'final state={state}  exit_type={exit_type}  bars={len(trace)}')
trace.tail()

In [ ]:
state_palette = {'CONFIRMATION': '#1f77b4', 'EXPANSION': '#2ca02c',
                  'MATURATION': '#ff7f0e', 'EXIT': '#d62728'}
fig, ax = plt.subplots(figsize=(12, 4))
for st, color in state_palette.items():
    sub = trace[trace['state'] == st]
    ax.scatter(sub['ts'], sub['price'], color=color, s=12, label=st)
ax.axhline(trade.entry_price, color='#888', linestyle=':', label='entry')
ax.axhline(trade.wall_price,  color='#cc1f1a', linestyle='--', label='stop (wall)')
ax.axhline(trade.current_target_price, color='#1f9d55', linestyle='--', label='target')
ax.set_title(f'Synthetic bounce trade — exit type = {exit_type}')
ax.set_xlabel('time (ms)'); ax.set_ylabel('price')
ax.legend(loc='upper left'); ax.grid(alpha=0.3); plt.show()

## 4. Exit-type distribution (Monte-Carlo over many synthetic trades)
Vary entry conditions and see which exit categories dominate.

In [ ]:
def run_one(seed, drift, vol):
    rng = np.random.default_rng(seed)
    prices = 100 + np.cumsum(rng.normal(drift, vol, 600))
    ts = np.arange(600) * 1_000
    trade = TradeCtx('BOUNCE', 'LONG', +1, prices[0], int(ts[0]),
                      prices[0] - 1.5, prices[0] + 3.0, 10.0, 0.01)
    for t, p in zip(ts, prices):
        ctx = RippleCtx(int(t), float(p), float(p),
                         cvd_slope=0.03 - 0.0002*(t/1000),
                         trade_rate=max(2.0, 10.0 - 0.02*(t/1000)),
                         impact=0.01 + 0.00002*(t/1000),
                         nearest_wall_depth=5.0, nearest_wall_quality=0.6)
        if check_invalidation(trade, ctx, cfg):  return 'INVALIDATION'
        if check_time(trade, ctx, cfg):          return 'TIME'
        if check_target(trade, ctx):             return 'TARGET'
        if check_exhaustion(trade, ctx, cfg):    return 'EXHAUSTION'
    return 'TIME'

results = []
for seed in range(400):
    for drift, vol in [(0.06, 0.5), (0.00, 0.5), (-0.04, 0.5), (0.0, 1.5)]:
        results.append({'drift': drift, 'vol': vol, 'exit': run_one(seed, drift, vol)})
rdf = pd.DataFrame(results)
rdf.groupby(['drift','vol'])['exit'].value_counts(normalize=True).unstack(fill_value=0)

## Takeaways

* The state machine is **monotone forward** under normal operation
  (no back-transitions other than EXPANSION ↔ MATURATION).
* Risk-budget and invalidation exits are *unconditional overrides*;
  they fire from any state.
* The exit-type histogram is a powerful diagnostic — a healthy
  strategy mixes TARGET and EXHAUSTION; pure TIME means the setup
  thresholds need recalibration.